In [1]:
# ==========================================================
# CELL 1 — IMPORTS AND PATHS
# ==========================================================

import pandas as pd
import sqlite3
from pathlib import Path

pd.set_option("display.max_columns", 100)

# Check where Jupyter is currently running
print("Current working directory:")
print(Path.cwd())

Current working directory:
c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\notebooks


In [2]:
# ==========================================================
# CELL 2 — FIND PROJECT DIRECTORIES
# ==========================================================

CURRENT_DIR = Path.cwd()

print("Current directory:", CURRENT_DIR)

# If notebook is inside notebooks/, project is one level above
if CURRENT_DIR.name.lower() == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SQL_DIR = BASE_DIR / "sql"

print("Base directory:", BASE_DIR)
print("Processed directory:", PROCESSED_DIR)
print("SQL directory:", SQL_DIR)

print("\nProcessed files:")

if PROCESSED_DIR.exists():
    for file in PROCESSED_DIR.glob("*.csv"):
        print(" -", file.name)
else:
    print("WARNING: processed folder not found")

Current directory: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\notebooks
Base directory: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics
Processed directory: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\data\processed
SQL directory: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\sql

Processed files:
 - accuracy_by_match_result.csv
 - group_standings.csv
 - matches.csv
 - matches_features.csv
 - match_prediction_results.csv
 - model_comparison.csv
 - players_features.csv
 - qualified_teams.csv
 - stadiums_cleaned.csv
 - teams_features.csv


In [3]:
# ==========================================================
# CELL 3 — CREATE SQL FOLDER
# ==========================================================

SQL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("SQL folder ready:")
print(SQL_DIR)

SQL folder ready:
c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\sql


In [4]:
# ==========================================================
# CELL 4 — LOAD FEATURE-ENGINEERED DATA
# ==========================================================

players = pd.read_csv(
    PROCESSED_DIR / "players_features.csv"
)

teams = pd.read_csv(
    PROCESSED_DIR / "teams_features.csv"
)

matches = pd.read_csv(
    PROCESSED_DIR / "matches_features.csv"
)

print("Players:", players.shape)
print("Teams:", teams.shape)
print("Matches:", matches.shape)

Players: (1248, 80)
Teams: (48, 137)
Matches: (104, 53)


In [5]:
# ==========================================================
# CELL 3 — CREATE SQLITE DATABASE
# ==========================================================

DATABASE_PATH = SQL_DIR / "football_analytics.db"

connection = sqlite3.connect(
    DATABASE_PATH
)

print("SQLite database created:")
print(DATABASE_PATH)

SQLite database created:
c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\sql\football_analytics.db


In [6]:
# ==========================================================
# CELL 4 — LOAD DATA INTO SQL TABLES
# ==========================================================

players.to_sql(
    "players",
    connection,
    if_exists="replace",
    index=False
)

teams.to_sql(
    "teams",
    connection,
    if_exists="replace",
    index=False
)

matches.to_sql(
    "matches",
    connection,
    if_exists="replace",
    index=False
)

print("Tables created successfully.")

Tables created successfully.


In [7]:
# ==========================================================
# CELL 5 — CHECK SQL TABLES
# ==========================================================

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)

display(tables)

,name
0,matches
1,players
2,teams


In [8]:
# ==========================================================
# CELL 6 — SQL QUERY: TOP SCORERS
# ==========================================================

query = """
SELECT
    player,
    team,
    position,
    goals,
    assists,
    minutes
FROM players
WHERE goals > 0
ORDER BY goals DESC, assists DESC
LIMIT 20;
"""

top_scorers_sql = pd.read_sql_query(
    query,
    connection
)

display(top_scorers_sql)

,player,team,position,goals,assists,minutes
0,Kylian Mbappé,France,FW,10.0,4.0,695.0
1,Lionel Messi,Argentina,FW,8.0,4.0,740.0
2,Jude Bellingham,England,MF,7.0,1.0,614.0
3,Erling Haaland,Norway,FW,7.0,0.0,465.0
4,Ousmane Dembélé,France,MF,6.0,2.0,592.0
5,Harry Kane,England,FW,6.0,1.0,652.0
6,Mikel Oyarzabal,Spain,FW,5.0,1.0,601.0
7,Vinicius Júnior,Brazil,FW,4.0,1.0,440.0
8,Julián Quiñones,Mexico,"FW,MF",4.0,1.0,410.0
9,Ismaila Sarr,Senegal,"MF,FW",4.0,1.0,364.0


In [9]:
# ==========================================================
# CELL 7 — SQL QUERY: TOP GOAL CONTRIBUTIONS
# ==========================================================

query = """
SELECT
    player,
    team,
    position,
    goals,
    assists,
    goal_contributions,
    minutes
FROM players
ORDER BY goal_contributions DESC
LIMIT 20;
"""

top_contributors_sql = pd.read_sql_query(
    query,
    connection
)

display(top_contributors_sql)

,player,team,position,goals,assists,goal_contributions,minutes
0,Kylian Mbappé,France,FW,10.0,4.0,14.0,695.0
1,Lionel Messi,Argentina,FW,8.0,4.0,12.0,740.0
2,Jude Bellingham,England,MF,7.0,1.0,8.0,614.0
3,Ousmane Dembélé,France,MF,6.0,2.0,8.0,592.0
4,Harry Kane,England,FW,6.0,1.0,7.0,652.0
5,Michael Olise,France,MF,0.0,7.0,7.0,646.0
6,Erling Haaland,Norway,FW,7.0,0.0,7.0,465.0
7,Bukayo Saka,England,MF,3.0,3.0,6.0,358.0
8,Mikel Oyarzabal,Spain,FW,5.0,1.0,6.0,601.0
9,Vinicius Júnior,Brazil,FW,4.0,1.0,5.0,440.0


In [10]:
# ==========================================================
# CELL 8 — SQL QUERY: ATTACKING EFFICIENCY
# ==========================================================

query = """
SELECT
    player,
    team,
    position,
    minutes,
    goals_per90,
    assists_per90,
    goal_contributions_per90
FROM players
WHERE minutes >= 180
ORDER BY goal_contributions_per90 DESC
LIMIT 20;
"""

efficient_attackers_sql = pd.read_sql_query(
    query,
    connection
)

display(efficient_attackers_sql)

,player,team,position,minutes,goals_per90,assists_per90,goal_contributions_per90
0,Johan Manzambi,Switzerland,"MF,FW",199.0,1.36,0.90,2.26
1,Kylian Mbappé,France,FW,695.0,1.29,0.52,1.81
2,Romelu Lukaku,Belgium,FW,233.0,1.16,0.39,1.55
3,Bukayo Saka,England,MF,358.0,0.75,0.75,1.50
4,Nathan Saliba,Canada,MF,182.0,0.49,0.99,1.48
5,Lionel Messi,Argentina,FW,740.0,0.97,0.49,1.46
6,Andreas Schjelderup,Norway,FW,252.0,0.36,1.07,1.43
7,Crysencio Summerville,Netherlands,FW,253.0,0.71,0.71,1.42
8,Erling Haaland,Norway,FW,465.0,1.35,0.00,1.35
9,Ismaila Sarr,Senegal,"MF,FW",364.0,0.99,0.25,1.24


In [11]:
# ==========================================================
# CELL 9 — SQL QUERY: DEFENSIVE LEADERS
# ==========================================================

query = """
SELECT
    player,
    team,
    position,
    minutes,
    tackles_won,
    interceptions,
    defensive_actions,
    defensive_actions_per90
FROM players
WHERE minutes >= 180
ORDER BY defensive_actions_per90 DESC
LIMIT 20;
"""

defensive_leaders_sql = pd.read_sql_query(
    query,
    connection
)

display(defensive_leaders_sql)

,player,team,position,minutes,tackles_won,interceptions,defensive_actions,defensive_actions_per90
0,Marvin Senaya,Ghana,DF,278.0,12.0,6.0,18.0,5.806
1,Rayan Aït-Nouri,Algeria,DF,284.0,13.0,5.0,18.0,5.625
2,Aurélien Tchouaméni,France,MF,360.0,14.0,6.0,20.0,5.000
3,Merchas Doski,Iraq,DF,270.0,9.0,6.0,15.0,5.000
4,Mohanad Lasheen,Egypt,MF,359.0,11.0,8.0,19.0,4.750
5,Khuliso Mudau,South Africa,DF,360.0,8.0,11.0,19.0,4.750
6,Livano Comenencia,Curaçao,MF,233.0,1.0,11.0,12.0,4.615
7,Mohamed Amine Ben Hamida,Tunisia,DF,202.0,9.0,1.0,10.0,4.545
8,Andrés Cubas,Paraguay,MF,480.0,13.0,11.0,24.0,4.528
9,Diego Gómez,Paraguay,MF,304.0,6.0,9.0,15.0,4.412


In [12]:
# ==========================================================
# CELL 10 — SQL QUERY: TEAM GOALS
# ==========================================================

query = """
SELECT
    team,
    games,
    goals,
    assists,
    shots,
    shots_on_target,
    goals_per90
FROM teams
ORDER BY goals DESC
LIMIT 20;
"""

team_goals_sql = pd.read_sql_query(
    query,
    connection
)

display(team_goals_sql)

,team,games,goals,assists,shots,shots_on_target,goals_per90
0,England,8,20,14,118,53,2.40
1,France,8,20,18,139,59,2.50
2,Argentina,8,18,12,114,44,2.00
3,Belgium,6,13,10,112,34,2.05
4,Spain,8,13,10,140,54,1.56
5,Norway,6,12,10,66,29,1.89
6,Germany,4,11,10,74,28,2.54
7,Brazil,5,10,8,74,30,2.00
8,Mexico,5,10,7,70,21,2.00
9,Morocco,6,10,9,69,26,1.58


In [13]:
# ==========================================================
# CELL 11 — SQL QUERY: TEAM SHOOTING EFFICIENCY
# ==========================================================

query = """
SELECT
    team,
    goals,
    shots,
    shots_on_target,
    goals_per_shot,
    shot_accuracy
FROM teams
WHERE shots > 0
ORDER BY goals_per_shot DESC
LIMIT 20;
"""

team_efficiency_sql = pd.read_sql_query(
    query,
    connection
)

display(team_efficiency_sql)

,team,goals,shots,shots_on_target,goals_per_shot,shot_accuracy
0,Japan,8,34,13,0.2353,0.3824
1,Netherlands,10,46,22,0.2174,0.4783
2,Norway,12,66,29,0.1818,0.4394
3,England,20,118,53,0.1695,0.4492
4,Croatia,6,37,17,0.1622,0.4595
5,Argentina,18,114,44,0.1579,0.3860
6,Austria,5,32,8,0.1562,0.2500
7,United States,9,59,19,0.1525,0.3220
8,Germany,11,74,28,0.1486,0.3784
9,Sweden,7,48,23,0.1458,0.4792


In [14]:
# ==========================================================
# CELL 12 — SQL QUERY: TEAM DEFENSIVE ACTIVITY
# ==========================================================

query = """
SELECT
    team,
    tackles_won,
    interceptions,
    defensive_actions
FROM teams
ORDER BY defensive_actions DESC
LIMIT 20;
"""

team_defense_sql = pd.read_sql_query(
    query,
    connection
)

display(team_defense_sql)

,team,tackles_won,interceptions,defensive_actions
0,Argentina,93,82,175
1,Spain,88,64,152
2,France,77,65,142
3,Paraguay,88,53,141
4,England,77,51,128
5,Norway,67,44,111
6,Switzerland,52,52,104
7,United States,42,60,102
8,Morocco,61,40,101
9,Egypt,54,44,98


In [15]:
# ==========================================================
# CELL 13 — SQL QUERY: MATCH RESULTS
# ==========================================================

query = """
SELECT
    result,
    COUNT(*) AS matches,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM matches),
        2
    ) AS percentage
FROM matches
GROUP BY result
ORDER BY matches DESC;
"""

match_results_sql = pd.read_sql_query(
    query,
    connection
)

display(match_results_sql)

,result,matches,percentage
0,Home Win,50,48.08
1,Away Win,30,28.85
2,Draw,24,23.08


In [16]:
# ==========================================================
# CELL 14 — SQL QUERY: HIGHEST-SCORING MATCHES
# ==========================================================

query = """
SELECT
    home_team,
    away_team,
    home_score,
    away_score,
    total_goals,
    result
FROM matches
ORDER BY total_goals DESC
LIMIT 15;
"""

highest_scoring_matches_sql = pd.read_sql_query(
    query,
    connection
)

display(highest_scoring_matches_sql)

,home_team,away_team,home_score,away_score,total_goals,result
0,France,England,4.0,6.0,10.0,Away Win
1,Germany,Curaçao,7.0,1.0,8.0,Home Win
2,Sweden,Tunisia,5.0,1.0,6.0,Home Win
3,England,Croatia,4.0,2.0,6.0,Home Win
4,Canada,Qatar,6.0,0.0,6.0,Home Win
5,Netherlands,Sweden,5.0,1.0,6.0,Home Win
6,Morocco,Haiti,4.0,2.0,6.0,Home Win
7,New Zealand,Belgium,1.0,5.0,6.0,Away Win
8,Algeria,Austria,3.0,3.0,6.0,Draw
9,United States,Paraguay,4.0,1.0,5.0,Home Win


In [17]:
# ==========================================================
# CELL 15 — SQL QUERY: POSSESSION VS RESULT
# ==========================================================

query = """
SELECT
    result,
    COUNT(*) AS matches,
    ROUND(
        AVG(home_possession),
        2
    ) AS avg_home_possession,
    ROUND(
        AVG(away_possession),
        2
    ) AS avg_away_possession
FROM matches
GROUP BY result;
"""

possession_results_sql = pd.read_sql_query(
    query,
    connection
)

display(possession_results_sql)

,result,matches,avg_home_possession,avg_away_possession
0,Away Win,30,43.47,56.67
1,Draw,24,55.96,44.04
2,Home Win,50,56.10,44.08


In [18]:
# ==========================================================
# CELL 16 — SQL QUERY: SHOT ADVANTAGE VS RESULT
# ==========================================================

query = """
SELECT
    result,
    COUNT(*) AS matches,
    ROUND(
        AVG(shot_difference),
        2
    ) AS avg_shot_difference,
    ROUND(
        AVG(shots_on_target_difference),
        2
    ) AS avg_sot_difference
FROM matches
GROUP BY result;
"""

shot_result_sql = pd.read_sql_query(
    query,
    connection
)

display(shot_result_sql)

,result,matches,avg_shot_difference,avg_sot_difference
0,Away Win,30,-2.60,-2.60
1,Draw,24,2.42,0.33
2,Home Win,50,7.16,3.84


In [19]:
# ==========================================================
# CELL 17 — SQL QUERY: POSITION ANALYSIS
# ==========================================================

query = """
SELECT
    position,
    COUNT(*) AS players,
    ROUND(AVG(goals), 2) AS avg_goals,
    ROUND(AVG(assists), 2) AS avg_assists,
    ROUND(AVG(goals_per90), 2) AS avg_goals_per90,
    ROUND(AVG(assists_per90), 2) AS avg_assists_per90
FROM players
GROUP BY position
ORDER BY avg_goals_per90 DESC;
"""

position_sql = pd.read_sql_query(
    query,
    connection
)

display(position_sql)

,position,players,avg_goals,avg_assists,avg_goals_per90,avg_assists_per90
0,"DF,MF",49,0.06,0.08,0.63,0.02
1,FW,169,0.70,0.28,0.26,0.13
2,"MF,FW",36,0.64,0.31,0.21,0.10
3,MF,378,0.28,0.28,0.12,0.11
4,"FW,MF",90,0.18,0.18,0.08,0.10
5,DF,364,0.08,0.10,0.03,0.04
6,"MF,DF",12,0.08,0.33,0.02,0.12
7,GK,145,0.00,0.00,0.00,0.00
8,"DF,FW",5,0.00,0.00,0.00,0.00


In [20]:
# ==========================================================
# CELL 18 — SQL JOIN: PLAYERS + TEAMS
# ==========================================================

query = """
SELECT
    p.player,
    p.position,
    p.goals,
    p.assists,
    p.goal_contributions,
    t.team,
    t.goals AS team_goals,
    t.possession
FROM players p
INNER JOIN teams t
    ON p.team = t.team
ORDER BY p.goal_contributions DESC
LIMIT 20;
"""

player_team_join_sql = pd.read_sql_query(
    query,
    connection
)

display(player_team_join_sql)

,player,position,goals,assists,goal_contributions,team,team_goals,possession
0,Kylian Mbappé,FW,10.0,4.0,14.0,France,20,55.9
1,Lionel Messi,FW,8.0,4.0,12.0,Argentina,18,57.6
2,Jude Bellingham,MF,7.0,1.0,8.0,England,20,54.1
3,Ousmane Dembélé,MF,6.0,2.0,8.0,France,20,55.9
4,Harry Kane,FW,6.0,1.0,7.0,England,20,54.1
5,Michael Olise,MF,0.0,7.0,7.0,France,20,55.9
6,Erling Haaland,FW,7.0,0.0,7.0,Norway,12,52.2
7,Bukayo Saka,MF,3.0,3.0,6.0,England,20,54.1
8,Mikel Oyarzabal,FW,5.0,1.0,6.0,Spain,13,64.1
9,Vinicius Júnior,FW,4.0,1.0,5.0,Brazil,10,53.0


In [21]:
# ==========================================================
# CELL 19 — SAVE SQL RESULTS
# ==========================================================

sql_results = {
    "top_scorers": top_scorers_sql,
    "top_contributors": top_contributors_sql,
    "efficient_attackers": efficient_attackers_sql,
    "defensive_leaders": defensive_leaders_sql,
    "team_goals": team_goals_sql,
    "team_efficiency": team_efficiency_sql,
    "team_defense": team_defense_sql,
    "match_results": match_results_sql,
    "highest_scoring_matches": highest_scoring_matches_sql,
    "possession_results": possession_results_sql,
    "shot_results": shot_result_sql,
    "position_analysis": position_sql,
    "player_team_join": player_team_join_sql
}

for name, dataframe in sql_results.items():

    dataframe.to_csv(
        SQL_DIR / f"{name}.csv",
        index=False
    )

print(
    f"Saved {len(sql_results)} SQL result files."
)

Saved 13 SQL result files.


In [22]:
# ==========================================================
# CELL 20 — SQL DATABASE VALIDATION
# ==========================================================

for table_name in [
    "players",
    "teams",
    "matches"
]:

    count_query = f"""
    SELECT COUNT(*) AS row_count
    FROM {table_name};
    """

    result = pd.read_sql_query(
        count_query,
        connection
    )

    print(
        f"{table_name}: "
        f"{result.loc[0, 'row_count']} rows"
    )

players: 1248 rows
teams: 48 rows
matches: 104 rows


In [23]:
# ==========================================================
# CELL 21 — CLOSE DATABASE
# ==========================================================

connection.close()

print("SQLite connection closed.")
print("SQL ANALYSIS COMPLETE")

SQLite connection closed.
SQL ANALYSIS COMPLETE
